In [1]:
from DSSATTools import (
    crop,
    WeatherStation,
    SoilProfile, SoilLayer,
    filex,
    DSSAT
)
import pandas as pd
from datetime import datetime
import tempfile
import os

/Users/ojiiik/Documents/Dayatani-Workingspace/dssat-simulation/.dssat_env/lib/python3.13/site-packages/DSSATTools/__init__.py:105: DeprecationWarning: DSSATTools version 3.0.0 is a major upgrade and will not be backwards compatible with previous versions. If you are running code that was developed  using a previous DSSATTools version, then install DSSATTools version 2.1.6
  warnings.warn(


### Weather

In [7]:
df = []
for year in range(19, 22):
    df.append(
        pd.read_csv(f"./weather_data/IDGR{year}01.csv")
    )

df = pd.concat(df, ignore_index=True)
df.columns = ['date', 'srad', 'tmax', 'tmin', 'rain']
df["date"] = pd.to_datetime(df["date"], format='%y%j')
df.head()

,date,srad,tmax,tmin,rain
0,2019-01-01,15.0,27.0,19.3,6.5
1,2019-01-02,19.4,27.8,19.7,16.4
2,2019-01-03,18.9,28.0,19.7,15.5
3,2019-01-04,22.4,28.3,19.9,4.2
4,2019-01-05,22.7,27.6,21.2,7.2


In [8]:
#Create weather station using the DataFrame
weather_station = WeatherStation(
    insi="IDGR", lat=-7.2245, long=107.9077, elev=0, tav=0, 
    amp=11.8, refht=2.0, wndht=2.0, table=df
)
weather_station

WeatherStation(insi='IDGR', lat=-7.2245, long=107.9077, elev=0.0, tav=0.0, amp=11.8, refht=2.0, wndht=2.0, cco2=nan, table=
   DATE  SRAD  TMAX  TMIN  RAIN  DEWP  WIND   PAR  EVAP  RHUM 
2019001  15.0  27.0  19.3   6.5   -99   -99   -99   -99   -99
2019002  19.4  27.8  19.7  16.4   -99   -99   -99   -99   -99
2019003  18.9  28.0  19.7  15.5   -99   -99   -99   -99   -99
2019004  22.4  28.3  19.9   4.2   -99   -99   -99   -99   -99
2019005  22.7  27.6  21.2   7.2   -99   -99   -99   -99   -99
2019006  21.9  28.2  20.9   6.5   -99   -99   -99   -99   -99
...
...)

### Soil
The soil profile can be created by entering all the soil surface properties, and all the layers' propeties:

In [16]:
# Import soil profile from a DSSAT soil file
import urllib.request
response = urllib.request.urlopen('https://github.com/DSSAT/dssat-csm-data/blob/develop/Soil/SOIL.SOL?raw=true')
soil_file_str = ''.join([l.decode('utf-8') for l in response])
with open('soil_data/SOIL.SOL', 'w') as f:
    f.write(soil_file_str)

In [22]:
# Create the soil profile from file
soil = SoilProfile.from_file('IBSG910085', 'soil_data/SOIL.SOL')
print(soil)

SoilProfile(name='IBSG910085', soil_data_source='IBSNAT', soil_clasification='', soil_depth=172.0, soil_series_name='Patencheru', site='Patancheru', country='India', lat=nan, long=nan, scs_family='ALFISOL Udic Rhodustalf, Patencheru Series', scom='', salb=0.13, slu1=6.0, sldr=0.3, slro=80.0, slnf=1.0, slpf=1.0, smhb='IB001', smpx='IB001', smke='IB001', table=
  SLB SLMH   SLLL  SDUL  SSAT  SRGF  SSKS  SBDM  SLOC  SLCL  SLSI  SLCF  SLNI  SLHW  SLHB  SCEC  SADC  SLPX  SLPT  SLPO CACO3  SLAL  SLFE  SLMN  SLBS  SLPA  SLPB  SLKE  SLMG  SLNA  SLSU  SLEC  SLCA 
   10 -99   0.080 0.220 0.310 1.000   -99  1.61  0.85   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99
   22 -99   0.090 0.220 0.310 0.900   -99  1.61  0.55   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99   -99
   52 -99   0.125 0.245 0.315 0.700   -99  

### Cultivar

In [24]:
cultivar = crop.Sorghum('IB0026')
cultivar

CropPars(var-name='CSH-1', expno='.', eco#=CropPars(econame='GENERIC', tbase=8.0, topt=34.0, ropt=34.0, gdde=6.0, rue=3.2, kcan=0.85, stpc=0.1, rtpc=0.25, tilfc=0.0, plam=6000.0), p1=410.0, p2=102.0, p2o=13.6, p2r=40.0, panth=617.5, p3=152.5, p4=81.5, p5=640.0, phint=49.0, g1=0.031, g2=5.982, pbase=nan, psat=nan)

In [31]:
# Define the Rice Cultivar

cultivar = crop.Rice('IB0003')
print(cultivar)

CropPars(var-name='IR 36', expno='.', eco#='IB0001', p1=556.8, p2r=53.88, p5=373.4, p2o=12.87, g1=68.0, g2=0.023, g3=1.0, phint=83.0, thot=31.3, tcldp=15.0, tcldf=15.0)


In [35]:
# Define Field. Note that the weather station and soil profile that we created before
# are one of the parameters to define the field
field = filex.Field(
    id_field='DTR00001', wsta=weather_station, flob=0, fldt='DR000', 
    fldd=0, flds=0, id_soil=soil
)
# Initial conditions is defined similar to the soil profile, including a table that 
# represents the soil profile.
initial_conditions = filex.InitialConditions(
    pcr='SG', icdat=datetime(1980, 7, 3), icrt=500, icnd=0,
    icrn=1, icre=1, icres=1300, icren=.5, icrep=0, icrip=100, icrid=10,
    table=pd.DataFrame([
        (10, .06, 2.5, 1.8),
        (22, .06, 2.5, 1.8),
        (52, .195, 3., 4.5),
        (82, .21, 3.5, 5.0),
        (112, 0.2, 2., 2.0),
        (142, 0.2, 1., 0.7),
        (172, 0.2, 1., 0.6),
    ], columns=['icbl', 'sh2o', 'snh4', 'sno3'])
)
# Define planting
planting = filex.Planting(
    pdate=datetime(2025, 5, 1), ppop=18, ppoe=18, plme='S',
    plds='R', plrs=45, plrd=0, pldp=5
)
# Define the simulation controls. The simulation controls section is split in subsections.
# Each sub-section of the simulation controls section has its own class
simulation_controls = filex.SimulationControls(
    general=filex.SCGeneral(sdate=datetime(1980, 6, 17)),
    options=filex.SCOptions(water='Y', nitro='Y', symbi='N'),
    methods=filex.SCMethods(infil='S'),
    management=filex.SCManagement(irrig='N', ferti='R', resid='N', harvs='M')
)

In [39]:
# Create the simulation environment
TMP = tempfile.tempdir 
dssat = DSSAT(os.path.join(TMP, 'dssat_test'))
# Run the model
results = dssat.run_treatment(
    field=field, cultivar=cultivar, planting=planting, 
    initial_conditions=initial_conditions, simulation_controls=simulation_controls
)

/var/folders/_w/2p93jxg90ngd07kf8mvrzvmw0000gp/T/dssat_test created.


OSError: [Errno 8] Exec format error: '/Users/ojiiik/Documents/Dayatani-Workingspace/dssat-simulation/.dssat_env/lib/python3.13/site-packages/DSSATTools/bin/dscsm048'

In [40]:
# DSSAT Error Diagnosis and Solution
import platform
import subprocess
from pathlib import Path

print("🔍 DSSAT Binary Compatibility Check")
print("=" * 50)

print(f"System: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Processor: {platform.processor()}")

is_apple_silicon = platform.machine() == 'arm64'
print(f"Apple Silicon: {'Yes ✅' if is_apple_silicon else 'No'}")

# Check DSSAT binary
try:
    from DSSATTools.run import BIN_PATH
    binary_path = Path(BIN_PATH)
    
    print(f"\nDSSAT Binary: {binary_path}")
    
    if binary_path.exists():
        print("✅ Binary exists")
        
        # Check binary architecture
        result = subprocess.run(['file', str(binary_path)], capture_output=True, text=True)
        print(f"Binary type: {result.stdout.strip()}")
        
        if is_apple_silicon and ('x86_64' in result.stdout or 'i386' in result.stdout):
            print("\n❌ PROBLEM IDENTIFIED:")
            print("   Intel x86_64 binary on Apple Silicon Mac")
            print("   This causes 'Exec format error'")
            
            print("\n🔧 SOLUTIONS:")
            print("1. Install Rosetta 2 (if not installed):")
            print("   softwareupdate --install-rosetta")
            
            print("\n2. Restart Jupyter with Rosetta:")
            print("   arch -x86_64 python -m jupyter notebook")
            
            print("\n3. Or run this specific notebook with Intel emulation")
            
            # Test Rosetta
            print("\n🧪 Testing Rosetta...")
            try:
                test = subprocess.run(['arch', '-x86_64', 'echo', 'Rosetta works'], 
                                    capture_output=True, text=True)
                if test.returncode == 0:
                    print("✅ Rosetta is available and working")
                else:
                    print("❌ Rosetta test failed")
            except:
                print("❌ Rosetta not available")
        else:
            print("✅ Binary architecture looks compatible")
    else:
        print("❌ Binary not found")
        
except Exception as e:
    print(f"Error checking DSSAT: {e}")

print(f"\n🎯 IMMEDIATE SOLUTION:")
print(f"   Close this notebook and restart with:")
print(f"   arch -x86_64 python -m jupyter notebook")
print(f"   Then reopen this notebook and run again.")

🔍 DSSAT Binary Compatibility Check
System: Darwin 25.0.0
Architecture: arm64
Processor: arm
Apple Silicon: Yes ✅

DSSAT Binary: /Users/ojiiik/Documents/Dayatani-Workingspace/dssat-simulation/.dssat_env/lib/python3.13/site-packages/DSSATTools/bin/dscsm048
✅ Binary exists
Binary type: /Users/ojiiik/Documents/Dayatani-Workingspace/dssat-simulation/.dssat_env/lib/python3.13/site-packages/DSSATTools/bin/dscsm048: ELF 64-bit LSB executable, x86-64, version 1 (GNU/Linux), statically linked, BuildID[sha1]=518f571a927295f5847524e805613045f69b4331, for GNU/Linux 3.2.0, not stripped
✅ Binary architecture looks compatible

🎯 IMMEDIATE SOLUTION:
   Close this notebook and restart with:
   arch -x86_64 python -m jupyter notebook
   Then reopen this notebook and run again.


## 🔧 DSSAT Error Solution

**Problem Identified:** 
- You're running on Apple Silicon (ARM64) Mac
- DSSAT binary is a Linux x86-64 ELF executable
- This causes "Exec format error" when trying to run

**Solution:**
1. **Install Rosetta 2** (if not already installed):
   ```bash
   softwareupdate --install-rosetta
   ```

2. **Restart Jupyter with Intel emulation:**
   ```bash
   arch -x86_64 python -m jupyter notebook
   ```

3. **Alternative: Run entire Python session with Rosetta:**
   ```bash
   arch -x86_64 /usr/bin/python3 -m pip install jupyter
   arch -x86_64 /usr/bin/python3 -m jupyter notebook
   ```

This will make the Intel x86-64 DSSAT binary run correctly on your Apple Silicon Mac using Rosetta 2 translation.